# RFP 요구사항 데이터셋 v0.2.0 EDA 노트북

이 노트북은 `data/processed/requirements_v0.2.0.jsonl` (총 1,024행) 동결 데이터셋의 문서별/유형별 분포, 본문 길이, 중첩표 및 특이 ID를 탐색하고 시각화합니다.

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# 한글 폰트 설정 (Windows/Linux/Mac 공통 지원 시도)
plt.rcParams['font.family'] = 'Malgun Gothic' if os.name == 'nt' else 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

dataset_path = Path('../data/processed/requirements_v0.2.0.jsonl')
if not dataset_path.exists():
    dataset_path = Path('data/processed/requirements_v0.2.0.jsonl')

df = pd.read_json(dataset_path, lines=True)
print(f"데이터셋 로드 완료: 총 {len(df):,} 행")
df.head(3)

In [ ]:
# 1. 문서별 요구사항 개수 집계
doc_counts = df['document_id'].value_counts().reset_index()
doc_counts.columns = ['document_id', 'count']
doc_counts['ratio_pct'] = (doc_counts['count'] / len(df) * 100).round(2)
display(doc_counts)

# 시각화
plt.figure(figsize=(10, 5))
sns.barplot(data=doc_counts, x='count', y='document_id', palette='viridis')
plt.title('문서별 요구사항 행 수 분포')
plt.xlabel('요구사항 수')
plt.ylabel('문서 ID')
plt.tight_layout()
plt.show()

In [ ]:
# 2. 요구사항 유형별 분포
type_counts = df['requirement_type'].value_counts().reset_index()
type_counts.columns = ['requirement_type', 'count']
display(type_counts)

# 시각화
plt.figure(figsize=(10, 4))
sns.barplot(data=type_counts, x='count', y='requirement_type', palette='magma')
plt.title('요구사항 유형별 분포')
plt.xlabel('행 수')
plt.ylabel('요구사항 유형')
plt.tight_layout()
plt.show()

In [ ]:
# 3. 본문 문자 길이 및 단어 수 파생 변수 생성
df['char_len'] = df['raw_requirement_text'].apply(len)
df['word_len'] = df['raw_requirement_text'].apply(lambda x: len(x.split()))

print("=== 문자 길이 통계 ===")
print(df['char_len'].describe(percentiles=[0.5, 0.9, 0.95, 0.99]))

# 히스토그램 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['char_len'], bins=40, kde=True, ax=axes[0], color='skyblue')
axes[0].set_title('본문 문자 길이 분포')
axes[0].set_xlabel('문자 수')

sns.boxplot(data=df, x='char_len', y='document_id', ax=axes[1], palette='coolwarm')
axes[1].set_title('문서별 본문 문자 길이 박스플롯')
axes[1].set_xlabel('문자 수')

plt.tight_layout()
plt.show()

In [ ]:
# 4. 중첩표(' | ') 및 특이 ID 검사
df['has_nested_table'] = df['raw_requirement_text'].apply(lambda x: ' | ' in x)
df['id_mismatch'] = df['requirement_id'] != df['source_requirement_id']

print(f"중첩표 포함 행 수: {df['has_nested_table'].sum()} 건")
print(f"Canonical ID != Source ID 불일치 행 수: {df['id_mismatch'].sum()} 건")

if df['id_mismatch'].sum() > 0:
    display(df[df['id_mismatch']][['requirement_uid', 'requirement_id', 'source_requirement_id', 'requirement_name']])